In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable,'../performance_analytics.py'],capture_output=True,text=True)
print(r.stdout[-4000:])
if r.returncode!=0: print('ERR:',r.stderr[-800:])

In [ ]:
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROC   = '../data/processed'
CHARTS = '../reports/charts'

def show(f):
    p=os.path.join(CHARTS,f)
    if os.path.exists(p): display(Image(p,width=900))
    else: print('Not found:',p)

scorecard = pd.read_csv(os.path.join(PROC,'fund_scorecard.csv'))
ab        = pd.read_csv(os.path.join(PROC,'alpha_beta.csv'))
print('Scorecard shape:', scorecard.shape)
print('Alpha/Beta shape:', ab.shape)
print(scorecard[['overall_rank','scheme_name','composite_score',
                  'cagr_3yr_pct','sharpe_ratio']].head(10).to_string(index=False))

# 📊 Performance Analytics — Bluestock Fintech MF Capstone
**Day 4 | Quantitative Fund Performance Analysis**

Covers: Daily returns · CAGR · Sharpe · Sortino · Alpha/Beta · Max Drawdown · Scorecard · Benchmark comparison

Risk-free rate: **Rf = 6.5%** (RBI repo rate proxy) | **252 trading days/year**


## 📈 Task 1 — Daily Returns Computation
**Formula:** `daily_return = NAV_t / NAV_t-1 − 1`

Validation: returns should be approximately normally distributed, centred near 0,
with equity funds showing ~1% daily σ and debt funds ~0.02% σ.


In [ ]:
show('chart_perf_01_return_distribution.png')

## 📊 Task 2 — CAGR Comparison Table
**Formula:** `CAGR = (NAV_end / NAV_start)^(1/n) − 1`

Periods: 1yr (252 days), 3yr (756 days), 5yr (1260 days) from latest NAV date.


In [ ]:
show('chart_perf_02_cagr_comparison.png')

## ⚡ Task 3 & 4 — Sharpe & Sortino Ratios
**Sharpe:** `(Rp − Rf) / σ(Rp) × √252`  |  Rf = 6.5% annual  
**Sortino:** `(Rp − Rf) / σ_downside(Rp) × √252` — uses only negative-return days

Higher is better. Sharpe ≥ 1.0 = good; ≥ 2.0 = excellent. Sortino penalises downside risk only.


In [ ]:
show('chart_perf_03_sharpe_sortino.png')

## 📐 Task 5 — Alpha & Beta (OLS Regression vs Nifty 100 TRI)
**Beta:** slope of `fund_return ~ nifty100_return` regression  
**Alpha (annualised):** `intercept × 252`  
**R²:** explains % of fund variance driven by market movement

Beta > 1 = more volatile than market | Alpha > 0 = outperforming benchmark


In [ ]:
show('chart_perf_04_alpha_beta.png')

## 📉 Task 6 — Maximum Drawdown Analysis
**Formula:** `MDD = min(NAV_t / running_max(NAV) − 1)`

Worst drawdown identifies the peak-to-trough loss period: start = last peak before trough,
end = trough date. Recovery date = first date NAV reclaims the prior peak.


In [ ]:
show('chart_perf_05_drawdown.png')

## 🏆 Task 7 — Fund Scorecard (0–100 Composite)
**Weights:**
- 30% × 3yr CAGR rank (higher = better)  
- 25% × Sharpe Ratio rank (higher = better)  
- 20% × Alpha rank (higher = better)  
- 15% × Expense Ratio rank (lower = better → inverse rank)  
- 10% × Max Drawdown rank (less negative = better → inverse rank)


In [ ]:
show('chart_perf_06_scorecard.png')

## 📊 Task 8 — Benchmark Comparison Chart (3yr) + Tracking Error
**Tracking Error:** `TE = std(fund_return − benchmark_return) × √252`

Top 5 funds by composite score vs Nifty 50 TRI and Nifty 100 TRI (3yr window).
Lower TE → closer index tracking | Higher TE → more active management.


In [ ]:
show('chart_perf_07_benchmark.png')

## 📋 Performance Analytics — Key Findings Summary

| Metric | Best Fund | Value | Category |
|--------|-----------|-------|----------|
| **Highest 3yr CAGR** | Quant Small Cap Fund Direct Gr | 25.44% | Equity |
| **Highest Sharpe Ratio** | Index Fund 34 Direct Growth | 1.378 | — |
| **Highest Sortino Ratio** | Index Fund 34 Direct Growth | 2.280 | — |
| **Highest Alpha (Ann.)** | DSP Top 100 Equity Fund Direct | 0.2172 | — |
| **Lowest Max Drawdown** | Nippon India Gilt Securities F | -2.91% | — |
| **Top Composite Score** | Index Fund 34 Direct Growth | 81.86/100 | — |

**Tracking Errors (Top 5 vs Nifty 100):** Range 15.17% – 21.49% (Active management confirmed)
